In [1]:
import numpy as np
import pandas as pd
import torch.nn as nn
from sklearn.preprocessing import OneHotEncoder
import torch.optim as optim
import warnings
import os
warnings.filterwarnings("ignore")

In [2]:
train_df = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')

In [3]:
df_encoded = pd.get_dummies(train_df, columns=train_df.select_dtypes(include=['object']).columns.tolist(), dtype=int)

In [4]:
df_encoded.head()

,id,age,study_hours,class_attendance,sleep_hours,exam_score,gender_female,gender_male,gender_other,course_b.com,...,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study,facility_rating_high,facility_rating_low,facility_rating_medium,exam_difficulty_easy,exam_difficulty_hard,exam_difficulty_moderate
0,0,21,7.91,98.8,4.9,78.3,1,0,0,0,...,0,0,1,0,0,1,0,1,0,0
1,1,18,4.95,94.8,4.7,46.7,0,0,1,0,...,0,0,0,1,0,0,1,0,0,1
2,2,20,4.68,92.6,5.8,99.0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,1
3,3,19,2.00,49.5,8.3,63.9,0,1,0,0,...,1,0,0,0,1,0,0,0,0,1
4,4,23,7.65,86.9,9.6,100.0,0,1,0,0,...,0,0,0,1,1,0,0,1,0,0


In [5]:
train_df['exam_difficulty'].unique()

array(['easy', 'moderate', 'hard'], dtype=object)

In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

class StudentTest(Dataset):
    def __init__(self, df):
        self.data = df.drop('id', axis=1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data.iloc[idx]
        if 'exam_score' in self.data.columns:
            X = sample.drop('exam_score').values
            y = sample['exam_score']
            return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
        else:
            return torch.tensor(sample.values, dtype=torch.float32)

In [30]:
train_ds = StudentTest(df_encoded)

In [8]:
df_encoded.iloc[100]

id                            100.00
age                            21.00
study_hours                     7.03
class_attendance               59.80
sleep_hours                     4.30
exam_score                     81.90
gender_female                   0.00
gender_male                     0.00
gender_other                    1.00
course_b.com                    0.00
course_b.sc                     1.00
course_b.tech                   0.00
course_ba                       0.00
course_bba                      0.00
course_bca                      0.00
course_diploma                  0.00
internet_access_no              0.00
internet_access_yes             1.00
sleep_quality_average           1.00
sleep_quality_good              0.00
sleep_quality_poor              0.00
study_method_coaching           0.00
study_method_group study        0.00
study_method_mixed              0.00
study_method_online videos      0.00
study_method_self-study         1.00
facility_rating_high            0.00
f

In [9]:
train_ds[0]

(tensor([21.0000,  7.9100, 98.8000,  4.9000,  1.0000,  0.0000,  0.0000,  0.0000,
          1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,
          1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,
          0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000]),
 tensor(78.3000))

In [10]:
class DNN(nn.Module):
    def __init__(self, in_features):
        super(DNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.ReLU(),
        )

    def forward(self, X):
        return self.layers(X)


model = DNN(in_features=len(train_ds[0][0]))

In [11]:
class RMSE(nn.Module):
    def __init__(self):
        super(RMSE, self).__init__()
        self.MSE = nn.MSELoss()
        self.eps = 1e-6
    def forward(self, y, target):
        loss = self.MSE(y, target)
        return torch.sqrt(loss)

In [12]:
#training
EPOCH = 10
BATCH = 1024
LEARNING_RATE = 0.01

train_loader = DataLoader(dataset=train_ds, batch_size=BATCH, shuffle=True)

In [13]:
len(train_ds)

630000

In [17]:
from tqdm import tqdm 

criterion = RMSE()
optimizer = optim.Adam(model.parameters(), lr=0.01)
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    NUM_WORKERS = torch.cuda.device_count() * min(4, os.cpu_count())
else: 
    NUM_WORKERS = min(4, os.cpu_count())

model = model.to(device)

for epoch in range(EPOCH):
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCH}')
    
    # Iterate over batches
    for batch_X, batch_y in loop:
        
        outputs = model(batch_X.to(device)) 
        loss = criterion(outputs, batch_y.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss to calculate average later
        running_loss += loss.item()
        
    
    # Calculate average loss for the epoch
    # len(train_loader) gives the number of batches (1000 / 32 ≈ 32 batches)
    avg_loss = running_loss / len(train_loader)
    
    if (epoch + 1) % 10 == 0:
        print(f"{epoch+1:<10} | {avg_loss:.4f}")

Epoch 10/10: 100%|██████████| 616/616 [03:32<00:00,  2.90it/s]

10         | 18.9216


In [31]:
test_df = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')
test_df_encoded = pd.get_dummies(test_df, columns=test_df.select_dtypes(include=['object']).columns.tolist(), dtype=int)
test_ds = StudentTest(test_df_encoded)
test_loader = DataLoader(dataset=test_ds, batch_size=BATCH, shuffle=False)

In [39]:
labels = []
for batch_X in test_loader:
    outputs = model(batch_X.to(device)) 
    labels.extend(outputs.cpu().detach().numpy().flatten())

In [47]:
submission = pd.DataFrame({
    'id': test_df_encoded['id'].values,
    'exam_score': labels
})

In [48]:
submission.to_csv('/kaggle/working/submission.csv' , index= False)